# Chapter 16a — Flash Attention: The Math (No GPU Required)

> Course: **llm.c — Zero to Hero**, Chapter 16a — the gentle **math prequel** to Chapter 16b.
> Builds on: **Chapter 6** (attention) and **Chapter 7** (softmax). If you can multiply two matrices, take an `exp`, and remember the chain rule from calculus, you have everything you need.
> Audience: **math freshman** · depth: first-principles, every step shown · code: **pure Python + numpy** (no CUDA).

Chapter 16b writes Flash Attention as a real CUDA kernel — fast, but it assumes you already *believe* the algorithm. This chapter earns that belief. We build Flash Attention's **forward** and **backward** pass from scratch, on paper and in numpy, until there is no magic left.

The punchline up front: **Flash Attention computes the exact same answer as ordinary attention — not an approximation — but it never builds the giant scores matrix.** It processes the keys in small blocks and carries a tiny running summary. Understanding *how* a running summary can equal the full computation is the whole chapter.

### What you'll learn

- What attention actually computes, on a tiny example you can check by hand.
- Why the `N×N` scores matrix is a memory disaster, and why we want to avoid ever storing it.
- **Safe softmax**: why we subtract the max before `exp`.
- **Online softmax**: how to compute a softmax one block at a time with a running max `m` and running denominator `ℓ` — and a proof it is *exact*.
- The **forward pass**: fold the values `V` in by keeping a running output and **rescaling** it as new blocks arrive.
- The **backward pass**: how to get gradients without storing anything `N×N`, using recomputation and the one-line trick `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)`.

### Sources grounding this chapter

- Zihao Ye — [*From Online Softmax to FlashAttention*](https://courses.cs.washington.edu/courses/cse599m/23sp/notes/flashattn.pdf) (UW CSE 599M, 2023). Our main step-by-step spine.
- [*Nobody has shown you the tiling algebra inside Flash Attention — here it is*](https://ai.plainenglish.io/nobody-has-shown-you-the-tiling-algebra-inside-flash-attention-here-it-is-5a9b15d48e7f) — the `α`/`β` rescaling view.
- [*The Flash Attention backward pass is the part nobody explains*](https://medium.com/data-and-beyond/the-flash-attention-backward-pass-is-the-part-nobody-explains-the-final-nail-in-the-coffin-9af72f15cea1) — the backward derivation.
- Milakov & Gimelshein — [*Online normalizer calculation for softmax*](https://arxiv.org/abs/1805.02867) (2018); Dao et al. — [*FlashAttention*](https://arxiv.org/abs/2205.14135) (2022), Algorithm 1 & Appendix B.


## 0. Setup — one import, and a 30-second matrix refresher

Everything runs in numpy. Two operations carry the whole chapter, so let's name them:

- **Dot product** of two length-`d` vectors `a · b = Σₖ aₖ bₖ` — one number measuring how aligned they are.
- **Matrix product** `A @ B`: entry `(i, j)` is the dot product of row `i` of `A` with column `j` of `B`.

In numpy, `@` is matrix multiply, `A.T` is transpose, and `axis=1` means "do it along each row."


In [ ]:
import numpy as np
np.random.seed(0)

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print("dot product a . b =", a @ b, " (= 1*4 + 2*5 + 3*6 = 32)")

A = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # shape (3, 2)
B = np.array([[2.0, 0.0, 1.0], [0.0, 2.0, 1.0]])     # shape (2, 3)
print("A @ B has shape", (A @ B).shape, "= (rows of A, cols of B)")
print(A @ B)


## 1. What attention actually computes

Attention takes three matrices, each with one row per token (sequence length `N`, head dimension `d`):

- **Q** (queries) — "what each token is looking for."
- **K** (keys) — "what each token offers."
- **V** (values) — "the information each token carries."

The computation is three lines:

$$ S = \frac{Q K^\top}{\sqrt{d}} \qquad P = \text{softmax}(S)\ \text{(row-wise)} \qquad O = P V $$

Read it left to right:

1. `S = QKᵀ/√d` — the **scores**. Entry `Sᵢⱼ` is the dot product of query `i` with key `j`: how much token `i` cares about token `j`. Shape `(N, N)`.
2. `P = softmax(S)` — turn each **row** of scores into a set of positive weights that sum to 1. So row `i` is "how token `i` divides its attention across all tokens."
3. `O = P V` — each output row is a **weighted average of the value vectors**, using those weights.

The `√d` just keeps the scores from getting huge when `d` is large; treat it as a fixed constant.


![A small attention-weight matrix; each row sums to 1](course/figures/fig_16a_scores_heatmap.png)

Above is a real `P` for `N=8` tokens. Notice every **row** sums to 1 — each query spreads exactly "one unit" of attention across the keys. Let's compute attention on a tiny example and verify by hand.


In [ ]:
def naive_softmax(x: np.ndarray) -> np.ndarray:
    '''Row-wise softmax: subtract each row's max (for stability), exponentiate, normalize.'''
    m = x.max(axis=1, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=1, keepdims=True)

def attention(Q, K, V):
    '''Textbook attention: O = softmax(QK^T / sqrt(d)) @ V.'''
    d = Q.shape[1]
    S = (Q @ K.T) / np.sqrt(d)      # (N, N) scores
    P = naive_softmax(S)            # (N, N) weights, each row sums to 1
    return P @ V                    # (N, d) outputs

# tiny example: 3 tokens, head dim 2
Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
V = np.array([[10.0, 0.0], [0.0, 20.0], [5.0, 5.0]])
O = attention(Q, K, V)
P = naive_softmax((Q @ K.T) / np.sqrt(2))
print("attention weights P (each row sums to 1):")
print(np.round(P, 3), " row sums:", P.sum(axis=1))
print("\noutput O (each row is a weighted average of the V rows):")
print(np.round(O, 3))


Look at output row 0. Its weights are roughly `[0.42, 0.21, 0.37]`, so its output is `0.42·[10,0] + 0.21·[0,20] + 0.37·[5,5]` — a blend of the three value vectors, tilted toward token 0 (which it matched most). That is *all* attention is: **a softmax-weighted average of value vectors.** Everything else in this chapter is about computing this exact result without paying the memory cost of the `(N, N)` matrix `P`.


## 2. The problem — the scores matrix `S` is the enemy

The matrices `Q`, `K`, `V`, `O` each have shape `(N, d)`. For a real model `d` is small (say 64 or 128) and fixed. But the scores `S` and weights `P` have shape `(N, N)` — they grow with the **square** of the sequence length.

That square is brutal. Watch the memory for *one* head's `N×N` scores in fp32 (4 bytes each):


In [ ]:
for N in [1024, 4096, 16384, 65536]:
    scores_bytes = N * N * 4              # the (N, N) matrix
    flash_bytes  = N * (64 + 2) * 4       # Flash keeps only o[d]+m+l per row -> O(N)
    print(f"N={N:6d}:  N x N scores = {scores_bytes/1e9:8.3f} GB   "
          f"vs  O(N) running state = {flash_bytes/1e6:6.2f} MB")


![Quadratic scores vs linear running state](course/figures/fig_16a_memory_blowup.png)

At `N = 65536` the scores matrix alone is **17 GB** — for a single head. Stacked over batches and heads it does not fit in any GPU. Meanwhile the only thing Flash Attention keeps around is, per output row, the running output `o` (length `d`) plus two numbers `m` and `ℓ` — that is `O(N)` total, the green line hugging the bottom.

So the goal is sharp: **compute `O = softmax(QKᵀ/√d) V` without ever holding a whole row of `S` at once.** The obstacle is the softmax — and the next two sections dismantle it.


## 3. Safe softmax — why subtract the max

The textbook softmax of a vector `x` is `softmax(x)ⱼ = e^{xⱼ} / Σₖ e^{xₖ}`. There is a landmine here: `e^x` overflows fast. In fp32, `e^{89}` is already near the largest representable number, and `e^{100}` is `inf`. One large score and the whole thing becomes `inf/inf = nan`.

The fix is an identity. For **any** constant `c`:

$$ \frac{e^{x_j}}{\sum_k e^{x_k}} = \frac{e^{x_j - c}}{\sum_k e^{x_k - c}} $$

because the `e^{-c}` cancels top and bottom. Choosing `c = max(x)` makes every exponent `≤ 0`, so every `e^{xⱼ - c} ∈ (0, 1]` — no overflow, ever. This is **safe softmax** (CSE 599M notes §2), and it is why `naive_softmax` above subtracted the row max.


In [ ]:
x = np.array([1.0, 2.0, 1000.0])    # one huge score

unsafe = np.exp(x) / np.exp(x).sum()                 # no max subtraction
m = x.max()
safe = np.exp(x - m) / np.exp(x - m).sum()           # subtract the max first

print("unsafe softmax:", unsafe, " <- nan, exp(1000) overflowed")
print("safe softmax:  ", safe,   " <- correct: ~[0, 0, 1]")
print("identity holds? subtracting the max changes nothing for small inputs:",
      np.allclose(np.exp(np.array([1.,2.,3.]))/np.exp(np.array([1.,2.,3.])).sum(),
                  naive_softmax(np.array([[1.,2.,3.]]))))


The unsafe version is `nan`; the safe one is correct. **Key takeaway:** to do a softmax you need the row's maximum `m` *before* you can exponentiate anything. But `m` is a property of the *whole* row — and we just said we refuse to hold the whole row. That tension is exactly what online softmax resolves.


## 4. Online softmax — a running max and a running denominator

Here is the trick (Milakov & Gimelshein 2018; CSE 599M §3). Walk through the row in **blocks**. Keep just two running numbers:

- `m` = the largest score seen *so far*,
- `ℓ` = the denominator `Σ e^{score − m}` over everything seen *so far*, measured relative to the current `m`.

When a new block arrives with its own local max `m̃`, two things can happen, and both are handled by one update. Let `mⁿᵉʷ = max(m, m̃)`. Then:

$$ \ell^{\text{new}} = \underbrace{e^{\,m - m^{\text{new}}}}_{\alpha}\,\ell \;+\; \underbrace{e^{\,\tilde m - m^{\text{new}}}}_{\beta}\,\tilde\ell $$

- `α = e^{m − mⁿᵉʷ}` **rescales the old denominator** onto the new reference max. If the new block raised the max, `α < 1` shrinks the old sum (it was measured against a smaller max, so it was relatively too big).
- `β = e^{m̃ − mⁿᵉʷ}` does the same for the new block's local sum `ℓ̃`.

Both factors are in `(0, 1]` because `mⁿᵉʷ` is at least as big as either max — so nothing ever overflows. Let's verify that processing a row in blocks gives the *exact* same softmax as doing it all at once.


In [ ]:
def online_softmax_stats(x_row: np.ndarray, block: int):
    '''Process one row in blocks, keeping running (m, l). Never sees the whole row at once.'''
    m, l = -np.inf, 0.0
    for j in range(0, len(x_row), block):
        xb = x_row[j:j + block]
        m_tilde = xb.max()
        m_new = max(m, m_tilde)
        alpha = np.exp(m - m_new) if np.isfinite(m) else 0.0   # rescale OLD denominator
        l = alpha * l + np.exp(xb - m_new).sum()               # beta is folded into exp(xb - m_new)
        m = m_new
    return m, l

row = np.random.randn(256) * 3.0           # 256 "scores", processed in blocks of 32
m, l = online_softmax_stats(row, block=32)
online_probs = np.exp(row - m) / l         # reconstruct the full softmax from (m, l) only

err = np.abs(online_probs - naive_softmax(row[None, :])[0]).max()
print(f"max difference vs all-at-once softmax: {err:.2e}  ->  {'PASS' if err < 1e-12 else 'FAIL'}")


The error is floating-point noise — **online softmax is exact, not approximate.** We computed a softmax over 256 numbers while only ever holding 32 at a time, by carrying two scalars `(m, ℓ)`. Now let's do *one* merge step entirely by hand so the `α`/`β` bookkeeping is concrete.


### 4a. One merge, by hand, with real numbers

Say we've processed block 1 and hold `m = 5`, `ℓ = 10`. Block 2 arrives with local max `m̃ = 7` and local denominator `ℓ̃ = 8`.

**Step 1 — new reference max.** `mⁿᵉʷ = max(5, 7) = 7`. The reference rose by 2.

**Step 2 — correction factors.**
$$ \alpha = e^{\,5 - 7} = e^{-2} \approx 0.135 \qquad \beta = e^{\,7 - 7} = 1.0 $$
The old state (built against max `5`) is shrunk by `α`; the new block already sits at the new max, so `β = 1`.

**Step 3 — combine.**
$$ \ell^{\text{new}} = 0.135 \cdot 10 + 1.0 \cdot 8 = 1.35 + 8 = 9.35 $$

That's the whole update. Below we run it, then *prove* it by building two raw blocks, summarizing each, merging, and comparing to a plain softmax over both blocks glued together.


In [ ]:
# the by-hand numbers
m, l = 5.0, 10.0
m_tilde, l_tilde = 7.0, 8.0
m_new = max(m, m_tilde)
alpha, beta = np.exp(m - m_new), np.exp(m_tilde - m_new)
l_new = alpha * l + beta * l_tilde
print(f"m_new={m_new}, alpha={alpha:.3f}, beta={beta:.1f}, l_new={l_new:.2f}  (expect 9.35)")

# now PROVE merging two real blocks == softmax over both blocks concatenated
np.random.seed(11)
block_a = np.random.randn(5) * 2
block_b = np.random.randn(5) * 2
def summarize(b):                       # one block's (local max, local denominator)
    mb = b.max(); return mb, np.exp(b - mb).sum()
ma, la = summarize(block_a)
mb, lb = summarize(block_b)
mn = max(ma, mb)
l_merged = np.exp(ma - mn) * la + np.exp(mb - mn) * lb
m_whole, l_whole = summarize(np.concatenate([block_a, block_b]))
print(f"merged (m, l) = ({mn:.3f}, {l_merged:.4f})")
print(f"whole  (m, l) = ({m_whole:.3f}, {l_whole:.4f})")
print("PASS" if np.isclose(l_merged, l_whole) and np.isclose(mn, m_whole) else "FAIL")


Merging two summaries equals summarizing the whole — that associativity is the mathematical heart of Flash Attention. The CSE 599M notes phrase the same idea as a recurrence over single elements; processing a *block* at a time is just the same recurrence applied to many elements at once.


## 5. The forward pass — fold in `V` and rescale the output

So far we tracked the softmax *denominator*. Attention also needs the numerator `Σ (weight · value)`. The fix is the same rescaling idea, now applied to a **running output vector** `O_acc` as well as `ℓ`.

Process the keys/values in blocks. Keep `(m, ℓ, O_acc)`. For each block:

1. Compute the score tile `S = Q·Kⱼᵀ/√d` for this block (it lives only here, then is thrown away).
2. `mⁿᵉʷ = max(m, rowmax(S))`, and `α = e^{m − mⁿᵉʷ}`.
3. `P = e^{S − mⁿᵉʷ}` (the new block's exponentiated scores, already at the new max — so its `β` is baked in).
4. `ℓ ← α·ℓ + rowsum(P)`  and  `O_acc ← α·O_acc + P·Vⱼ`.

Notice `O_acc` is the **unnormalized** numerator — we do *not* divide by `ℓ` yet. Only at the very end do we return `O = O_acc / ℓ`. This is **deferred normalization**: dividing early would just have to be undone when the next block grows `ℓ`.


![Block by block, the running output rescales onto the exact answer](course/figures/fig_16a_online_convergence.png)

The plot shows one output coordinate as blocks stream in: each block nudges and **rescales** the running `O_acc/ℓ` until it lands exactly on the full-row answer (dashed line). Here is the whole forward pass in numpy — the algorithm completely naked:


In [ ]:
def flash_forward(Q, K, V, block: int):
    '''Exact attention computed block-by-block. The (N, N) scores never exist:
    only a (N, block) tile is alive at any moment. State per row: o[d], m, l.'''
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O_acc = np.zeros((N, d))            # running UNNORMALIZED output (the numerator)
    m = np.full((N, 1), -np.inf)        # running max per row
    l = np.zeros((N, 1))                # running denominator per row
    for j in range(0, N, block):        # loop over key/value blocks
        Kj, Vj = K[j:j + block], V[j:j + block]
        S = (Q @ Kj.T) * scale          # (N, block) score tile -- the ONLY big thing, and it's small
        m_tilde = S.max(axis=1, keepdims=True)
        m_new = np.maximum(m, m_tilde)
        alpha = np.exp(m - m_new)        # rescale factor for old running state
        P = np.exp(S - m_new)            # (N, block); new block already at the new max
        l = alpha * l + P.sum(axis=1, keepdims=True)
        O_acc = alpha * O_acc + P @ Vj   # rescale old numerator, add this block
        m = m_new
    return O_acc / l                     # deferred normalization: divide ONCE at the end

N, d = 128, 16
Q, K, V = (np.random.randn(N, d) for _ in range(3))
O_exact = attention(Q, K, V)             # textbook, builds the full (N, N) matrix
O_flash = flash_forward(Q, K, V, block=16)
err = np.abs(O_flash - O_exact).max()
print(f"flash forward vs textbook attention, max difference: {err:.2e}  ->  {'PASS' if err < 1e-10 else 'FAIL'}")


Bit-for-bit (up to floating-point noise) the **same** answer as textbook attention — but the biggest array we ever allocated was the `(128, 16)` score *tile*, never the `(128, 128)` full matrix. That is the entire forward pass of Flash Attention. The CUDA kernel in Chapter 16b is this exact loop, with the blocks living in fast on-chip memory and one GPU thread per row.


## 6. The tiling picture (where this runs on a GPU)

We never said "GPU" in the math — and we didn't need to. But the reason this shape matters is the memory hierarchy: a GPU has a little bit of very fast **on-chip** memory (SRAM) and a lot of slow **off-chip** memory (HBM). Flash Attention keeps the running `(m, ℓ, O_acc)` and one small block in SRAM, and touches slow memory as little as possible.

```mermaid
flowchart LR
  subgraph SLOW["slow memory (HBM) - big"]
    Q[Q rows]
    KV[K, V blocks]
    OUT[O output]
  end
  subgraph FAST["fast memory (SRAM) - small, where the work happens"]
    direction TB
    tile["one score tile S = Qi Kj^T<br/>(computed, used, thrown away)"]
    state["running state per row:<br/>m, l, O_acc"]
  end
  Q --> tile
  KV --> tile
  tile --> state
  state -->|after last block: O_acc / l| OUT
```

Each block does: load a small `K/V` block → compute its score tile → update the running `(m, ℓ, O_acc)` → discard the tile. The `(N, N)` scores never touch slow memory because they are never assembled. *That* is why Flash Attention is fast — it is the same arithmetic, but it moves far less memory. (Chapter 16b makes this concrete in CUDA and measures the speedup.)


## 7. The backward pass — gradients without the `N×N` matrices

Training a model means nudging `Q`, `K`, `V` to lower a loss. Backpropagation hands us `dO` (how the loss changes with each entry of the output `O`) and asks us to produce `dQ`, `dK`, `dV`. The trouble: the textbook backward pass needs the `(N, N)` matrices `P` and `dP` — exactly what we refused to store.

Two ideas rescue it.

**Idea 1 — recomputation.** In the forward pass we saved only the `O(N)` per-row stats `(m, ℓ)`. In the backward pass we **recompute** the score tile `S` and the weights `P` block-by-block from the saved `Q, K, V` and those stats — same tiling as the forward. We redo some arithmetic, but we never read an `N×N` matrix from slow memory. (Production Flash Attention saves a single per-row number `Lᵢ = mᵢ + log ℓᵢ`, the log-sum-exp, instead of `m` and `ℓ` separately — same idea, one scalar.)

**Idea 2 — the `D` trick.** The gradient of a softmax looks expensive, but it collapses to one number per row.


### 7a. The gradient identities

With `O = P V` and `P = softmax(S)` row-wise, reverse-mode differentiation gives (backward-pass blog; paper Appendix B):

$$ dV = P^\top dO \qquad dP = dO\,V^\top \qquad dS = P \circ \big(dP - \text{rowsum}(P \circ dP)\big) $$

(`∘` is elementwise product; `rowsum` sums each row to one number.) Then, because `S = QKᵀ·\text{scale}`:

$$ dQ = (dS\,K)\cdot\text{scale} \qquad dK = (dS^\top Q)\cdot\text{scale} $$

The one expensive-looking piece is `rowsum(P ∘ dP)`. Define

$$ D_i = \text{rowsum}(dO_i \circ O_i) $$

— a single number per row, built from the **output and its gradient**, both of which we have. The claim is `rowsum(P ∘ dP) = D`. Here is the one-line proof for row `i`, using `Oᵢ = Σⱼ Pᵢⱼ Vⱼ` and `dPᵢⱼ = dOᵢ · Vⱼ`:

$$ \sum_j P_{ij}\,dP_{ij} = \sum_j P_{ij}\,(dO_i \cdot V_j) = dO_i \cdot \Big(\sum_j P_{ij} V_j\Big) = dO_i \cdot O_i = D_i. $$

So `dS = P ∘ (dP − D)`, and `D` needs no `N×N` storage at all. Let's verify the whole backward pass two ways: against a textbook materialized backward, and against finite differences (the ground truth — the definition of a derivative).


In [ ]:
def backward_textbook(Q, K, V, dO):
    '''Reference backward: build the full (N, N) P and dP, use the softmax Jacobian directly.'''
    d = Q.shape[1]; scale = 1.0 / np.sqrt(d)
    S = (Q @ K.T) * scale
    P = naive_softmax(S)
    O = P @ V
    dV = P.T @ dO
    dP = dO @ V.T
    dS = P * (dP - (P * dP).sum(axis=1, keepdims=True))   # rowsum(P o dP) spelled out
    dQ = (dS @ K) * scale
    dK = (dS.T @ Q) * scale
    return dQ, dK, dV, O

def backward_flash(Q, K, V, O, dO, block):
    '''Flash-style: recompute P block-by-block from saved (m, l); use D = rowsum(dO o O).
    Never stores an (N, N) matrix.'''
    N, d = Q.shape; scale = 1.0 / np.sqrt(d)
    D = (dO * O).sum(axis=1, keepdims=True)               # the trick: one scalar per row
    # the O(N) stats the forward pass saved (max m and denominator l, per row):
    full = (Q @ K.T) * scale
    m = full.max(axis=1, keepdims=True)
    l = np.exp(full - m).sum(axis=1, keepdims=True)
    dQ = np.zeros_like(Q); dK = np.zeros_like(K); dV = np.zeros_like(V)
    for j in range(0, N, block):                          # block over keys/values
        Kj, Vj = K[j:j + block], V[j:j + block]
        S = (Q @ Kj.T) * scale                            # recomputed tile (N, block)
        P = np.exp(S - m) / l                              # exact weights for this tile
        dV[j:j + block] = P.T @ dO
        dP = dO @ Vj.T                                     # (N, block)
        dS = P * (dP - D)                                  # uses D, never a full rowsum
        dQ += (dS @ Kj) * scale
        dK[j:j + block] = (dS.T @ Q) * scale
    return dQ, dK, dV

np.random.seed(1)
N, d = 96, 16
Q, K, V = (np.random.randn(N, d) for _ in range(3))
dO = np.random.randn(N, d)
dQt, dKt, dVt, O = backward_textbook(Q, K, V, dO)
dQf, dKf, dVf = backward_flash(Q, K, V, O, dO, block=16)
for name, f, t in [("dQ", dQf, dQt), ("dK", dKf, dKt), ("dV", dVf, dVt)]:
    e = np.abs(f - t).max()
    print(f"{name}: flash vs textbook max diff {e:.2e}  ->  {'PASS' if e < 1e-9 else 'FAIL'}")


In [ ]:
# Independent ground truth: finite differences. For a scalar loss = sum(W * O),
# the upstream gradient is exactly dO = W, and d(loss)/d(Q[i,k]) ~ (loss(Q+eps) - loss(Q-eps)) / 2eps.
W = np.random.randn(N, d)
_, _, _, O0 = backward_textbook(Q, K, V, W)
dQ_analytic, _, _ = backward_flash(Q, K, V, O0, W, block=16)

def loss(Qx):
    return (W * attention(Qx, K, V)).sum()

eps = 1e-5
dQ_fd = np.zeros_like(Q)
for i in range(N):
    for k in range(d):
        Qp = Q.copy(); Qp[i, k] += eps
        Qm = Q.copy(); Qm[i, k] -= eps
        dQ_fd[i, k] = (loss(Qp) - loss(Qm)) / (2 * eps)
e = np.abs(dQ_analytic - dQ_fd).max()
print(f"dQ: flash-analytic vs finite-difference max diff {e:.2e}  ->  {'PASS' if e < 1e-5 else 'FAIL'}")


Both checks pass. The Flash backward pass — recompute `P` block-by-block, use `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)` — produces the **exact** gradients with only `O(N)` saved state. The finite-difference check is the real judge: it never saw any of our algebra, it just wiggled inputs and measured the output, and it agrees. The math is sound.


## 8. Exercises

Five exercises, easy to harder. Each has a runnable check; **predict the result before you run it**, then click to reveal the solution.


### Exercise 1 — softmax is shift-invariant

Show numerically that adding *any* constant `c` to every entry of a vector leaves its softmax unchanged. Fill in the TODO, then predict whether the difference is zero.


In [ ]:
x = np.array([[2.0, 4.0, 1.0, 3.0]])
c = 100.0
# TODO: build x_shifted by adding c to every entry of x
x_shifted = x + c   # replace
diff = np.abs(naive_softmax(x) - naive_softmax(x_shifted)).max()
print(f"max difference: {diff:.2e}  ->  {'PASS' if diff < 1e-12 else 'FAIL (still the stub)'}")


<details>
<summary>▶ Show solution</summary>

```python
x_shifted = x + c   # softmax(x + c) == softmax(x): the e^c cancels top and bottom
diff = np.abs(naive_softmax(x) - naive_softmax(x_shifted)).max()
print(f"max difference: {diff:.2e}  ->  PASS")
```

</details>

### Exercise 2 — merge two softmax summaries

The core primitive: given `(m_a, ℓ_a)` and `(m_b, ℓ_b)` for two disjoint chunks of the same row, return the combined `(m, ℓ)` — without re-seeing the raw scores. Fill in the TODO.


In [ ]:
def merge(m_a, l_a, m_b, l_b):
    # TODO: pick the new max, then rescale BOTH denominators onto it before adding
    m_new = max(m_a, m_b)   # replace
    l_new = np.exp(m_a - m_new) * l_a + np.exp(m_b - m_new) * l_b   # replace
    return m_new, l_new

np.random.seed(7)
row = np.random.randn(200) * 2.5
ma, la = online_softmax_stats(row[:100], block=25)
mb, lb = online_softmax_stats(row[100:], block=25)
m_merged, l_merged = merge(ma, la, mb, lb)
m_whole, l_whole = online_softmax_stats(row, block=25)
ok = np.isclose(m_merged, m_whole) and np.isclose(l_merged, l_whole)
print("PASS" if ok else "FAIL (still the stub)")


<details>
<summary>▶ Show solution</summary>

```python
def merge(m_a, l_a, m_b, l_b):
    m_new = max(m_a, m_b)
    l_new = np.exp(m_a - m_new) * l_a + np.exp(m_b - m_new) * l_b
    return m_new, l_new

m_merged, l_merged = merge(ma, la, mb, lb)
m_whole, l_whole = online_softmax_stats(row, block=25)
assert np.isclose(m_merged, m_whole) and np.isclose(l_merged, l_whole)
print("PASS")
```

</details>

### Exercise 3 — does block size change the answer?

`flash_forward` takes a `block` argument. **Predict:** does the final output depend on the block size? Run it with several block sizes and compare to textbook attention.


In [ ]:
N, d = 64, 8
Q, K, V = (np.random.randn(N, d) for _ in range(3))
O_exact = attention(Q, K, V)
for block in [1, 8, 16, 64]:
    err = np.abs(flash_forward(Q, K, V, block) - O_exact).max()
    print(f"block={block:3d}:  max diff vs textbook = {err:.2e}")
# TODO: in one line, print whether ALL block sizes match (err < 1e-10)


<details>
<summary>▶ Show solution</summary>

```python
# The answer is the SAME for every block size -- block size only affects memory/speed,
# never correctness, because merging summaries is exact (associative).
print("all match:", all(np.abs(flash_forward(Q, K, V, b) - O_exact).max() < 1e-10
                        for b in [1, 8, 16, 64]))
```

</details>

### Exercise 4 — compute `D` two ways

The backward pass claims `rowsum(P ∘ dP) = rowsum(dO ∘ O) = D`. Verify it numerically: compute `D` the cheap way (from `dO` and `O`) and the expensive way (build `P` and `dP`), and confirm they match.


In [ ]:
np.random.seed(2)
N, d = 32, 8
Q, K, V = (np.random.randn(N, d) for _ in range(3))
dO = np.random.randn(N, d)
_, _, _, O = backward_textbook(Q, K, V, dO)

D_cheap = (dO * O).sum(axis=1, keepdims=True)          # the trick: only O and dO
# TODO: compute D_expensive = rowsum(P o dP) by building P and dP explicitly
P = naive_softmax((Q @ K.T) / np.sqrt(d))
dP = dO @ V.T
D_expensive = (P * dP).sum(axis=1, keepdims=True)   # replace with rowsum(P * dP)
print("match?", np.allclose(D_cheap, D_expensive), "->",
      "PASS" if np.allclose(D_cheap, D_expensive) and D_expensive.any() else "FAIL (still the stub)")


<details>
<summary>▶ Show solution</summary>

```python
D_expensive = (P * dP).sum(axis=1, keepdims=True)
assert np.allclose(D_cheap, D_expensive)
print("match? True -> PASS  (this is why we never need to store P or dP)")
```

</details>

### Exercise 5 — a causal mask in the streaming forward

Autoregressive models forbid token `i` from attending to future tokens `j > i`. In `flash_forward`, a key's **global** index is `j_start + c` for column `c` of the current block. Mask future keys by setting their score to `-∞` (so `e^{...} = 0`). Complete the masked forward and check it against a masked textbook reference.


In [ ]:
def attention_causal(Q, K, V):
    d = Q.shape[1]
    S = (Q @ K.T) / np.sqrt(d)
    mask = np.triu(np.ones_like(S), k=1).astype(bool)   # True where j > i (future)
    S[mask] = -np.inf
    return naive_softmax(S) @ V

def flash_forward_causal(Q, K, V, block):
    N, d = Q.shape; scale = 1.0 / np.sqrt(d)
    O_acc = np.zeros((N, d)); m = np.full((N, 1), -np.inf); l = np.zeros((N, 1))
    rows = np.arange(N)[:, None]
    for j in range(0, N, block):
        Kj, Vj = K[j:j + block], V[j:j + block]
        S = (Q @ Kj.T) * scale
        cols = (j + np.arange(Kj.shape[0]))[None, :]
        # TODO: wherever cols > rows (a future key), set that score S to -np.inf
        print(cols)
        S[cols > rows] = -np.inf
        m_new = np.maximum(m, S.max(axis=1, keepdims=True))
        alpha = np.exp(m - m_new); P = np.exp(S - m_new)
        l = alpha * l + P.sum(axis=1, keepdims=True)
        O_acc = alpha * O_acc + P @ Vj
        m = m_new
    return O_acc / l

N, d = 48, 8
Q, K, V = (np.random.randn(N, d) for _ in range(3))
err = np.abs(flash_forward_causal(Q, K, V, 8) - attention_causal(Q, K, V)).max()
print(f"max diff vs causal textbook: {err:.2e}  ->  {'PASS' if err < 1e-10 else 'FAIL (still the stub)'}")


<details>
<summary>▶ Show solution</summary>

```python
# add one line inside the loop, right after computing S:
#     S = np.where(cols > rows, -np.inf, S)
def flash_forward_causal(Q, K, V, block):
    N, d = Q.shape; scale = 1.0 / np.sqrt(d)
    O_acc = np.zeros((N, d)); m = np.full((N, 1), -np.inf); l = np.zeros((N, 1))
    rows = np.arange(N)[:, None]
    for j in range(0, N, block):
        Kj, Vj = K[j:j + block], V[j:j + block]
        S = (Q @ Kj.T) * scale
        cols = (j + np.arange(Kj.shape[0]))[None, :]
        S = np.where(cols > rows, -np.inf, S)            # mask future keys
        m_new = np.maximum(m, S.max(axis=1, keepdims=True))
        alpha = np.exp(m - m_new); P = np.exp(S - m_new)
        l = alpha * l + P.sum(axis=1, keepdims=True)
        O_acc = alpha * O_acc + P @ Vj
        m = m_new
    return O_acc / l

err = np.abs(flash_forward_causal(Q, K, V, 8) - attention_causal(Q, K, V)).max()
assert err < 1e-10
print(f"max diff {err:.2e} -> PASS")
```

</details>

## Further reading

**Source of truth**
- Zihao Ye, [*From Online Softmax to FlashAttention*](https://courses.cs.washington.edu/courses/cse599m/23sp/notes/flashattn.pdf) — the recurrence-based build-up this chapter follows.
- Dao et al., [*FlashAttention*](https://arxiv.org/abs/2205.14135) (2022) — Algorithm 1 (forward), Appendix B (backward), Theorems 1–2 (memory & IO).

**Going deeper**
- Milakov & Gimelshein, [*Online normalizer calculation for softmax*](https://arxiv.org/abs/1805.02867) (2018) — where online softmax comes from.
- [*Tiling algebra inside Flash Attention*](https://ai.plainenglish.io/nobody-has-shown-you-the-tiling-algebra-inside-flash-attention-here-it-is-5a9b15d48e7f) — the `α`/`β` rescaling view.
- [*The Flash Attention backward pass*](https://medium.com/data-and-beyond/the-flash-attention-backward-pass-is-the-part-nobody-explains-the-final-nail-in-the-coffin-9af72f15cea1) — the gradient derivation.
- **Chapter 16b — Flash Attention from Scratch** (this course): the same algorithm as a real CUDA kernel, with IO-complexity proofs and FA-1 vs FA-2.


## Recap

- Attention is `O = softmax(QKᵀ/√d) V` — a softmax-weighted average of value vectors.
- The villain is the `(N, N)` scores matrix `S`: quadratic memory, the thing we must never store.
- **Safe softmax** subtracts the row max so `exp` can't overflow — but it needs the whole-row max first.
- **Online softmax** removes that obstacle: process blocks, carry a running max `m` and denominator `ℓ`, rescale with `α = e^{m−mⁿᵉʷ}`. It is **exact**.
- **Forward pass:** also carry a running unnormalized output `O_acc`, rescale it the same way, divide by `ℓ` once at the end (deferred normalization). Same answer as textbook attention, no `N×N` matrix.
- **Backward pass:** recompute `P` block-by-block from the saved `O(N)` stats, and collapse the softmax gradient with `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)`. Verified against finite differences.

Next: **Chapter 16b** turns this exact loop into a CUDA kernel — blocks in on-chip SRAM, one thread per query row — and measures why touching less memory makes it dramatically faster.
